# Deep (stacked) RNNs
1. A two-layer SimpleRNN on the toy reviews: parameter count and a hand forward pass checked against Keras.
2. Stacked SimpleRNN, LSTM and GRU on IMDB in Keras: parameter counts, and why return_sequences=True is needed.
3. Experiment: next-character prediction on IMDB review text with 1 LSTM layer, 2 stacked LSTM layers,
   and 1 wider LSTM layer with about the same number of parameters as the stack. 2 seeds each.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()   # same numbers on every run
print("GPU:", tf.config.list_physical_devices("GPU"))
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## 1. Two stacked SimpleRNN layers on the toy reviews
Vocabulary cat, mat, rat (one-hot, 3 numbers). Layer 1 has 3 nodes, layer 2 has 2 nodes, then 1 sigmoid node.

In [ ]:
vocab = ["cat", "mat", "rat"]
one_hot = {w: np.eye(3)[i] for i, w in enumerate(vocab)}
reviews = ["cat mat rat", "rat rat mat", "mat mat cat"]
X = np.array([[one_hot[w] for w in r.split()] for r in reviews], dtype="float32")
print("batch shape:", X.shape)

toy = keras.Sequential([keras.Input(shape=(3, 3)),
                        keras.layers.SimpleRNN(3, return_sequences=True),   # layer 1: hands every h_t up
                        keras.layers.SimpleRNN(2),                          # layer 2: only its last h_T
                        keras.layers.Dense(1, activation="sigmoid")])
toy.summary()
counts = [l.count_params() for l in toy.layers]
print("parameters per layer:", counts)
assert counts == [3 * 3 + 3 * 3 + 3, 3 * 2 + 2 * 2 + 2, 2 + 1]

In [ ]:
# Hand forward pass: h_t^(l) = tanh(h_t^(l-1) W_i^(l) + h_{t-1}^(l) W_h^(l) + b^(l)), with h_t^(0) = x_t
Wi1, Wh1, b1, Wi2, Wh2, b2, Wy, by = toy.get_weights()


def stacked_forward(x):
    h1, h2 = np.zeros(3), np.zeros(2)
    for x_t in x:
        h1 = np.tanh(x_t @ Wi1 + h1 @ Wh1 + b1)     # layer 1 sees the word
        h2 = np.tanh(h1 @ Wi2 + h2 @ Wh2 + b2)      # layer 2 sees layer 1's hidden state at the same t
    return 1 / (1 + np.exp(-(h2 @ Wy + by)))


by_hand = np.array([stacked_forward(x) for x in X]).ravel()
by_keras = toy.predict(X, verbose=0).ravel()
print("by hand:", by_hand.round(4), " Keras:", by_keras.round(4))
assert np.allclose(by_hand, by_keras, atol=1e-5)

## 2. Stacked layers on IMDB in Keras: parameter counts

In [ ]:
def stacked(layer, units=5):
    return keras.Sequential([keras.Input(shape=(100,)),
                             keras.layers.Embedding(10000, 32),
                             layer(units, return_sequences=True),
                             layer(units),
                             keras.layers.Dense(1, activation="sigmoid")])


rows = []
for name, layer in (("SimpleRNN", keras.layers.SimpleRNN), ("LSTM", keras.layers.LSTM), ("GRU", keras.layers.GRU)):
    m = stacked(layer)
    c = [l.count_params() for l in m.layers]
    rows.append(dict(model="deep " + name, embedding=c[0], layer_1=c[1], layer_2=c[2], dense=c[3], total=sum(c)))
params = pd.DataFrame(rows)
print(params.to_string(index=False))
assert list(params.loc[0, ["embedding", "layer_1", "layer_2", "dense"]]) == [320000, 190, 55, 6]
params.to_csv(DATA / "imdb_params.csv", index=False)

In [ ]:
# Without return_sequences=True on the lower layer, the upper layer gets only one vector per review, not a sequence
try:
    keras.Sequential([keras.Input(shape=(100,)), keras.layers.Embedding(10000, 32),
                      keras.layers.SimpleRNN(5), keras.layers.SimpleRNN(5)])
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0][:160])
print("output shape of SimpleRNN(5, return_sequences=True):",
      keras.layers.SimpleRNN(5, return_sequences=True)(np.zeros((1, 100, 32), "float32")).shape)

## 3. Experiment: does a second layer help on a complex task with lots of data?
Task: read IMDB review text one character at a time and predict the next character at every step.
Training text: 3 million characters from the training reviews; validation: 300,000 from the test reviews.

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.imdb.load_data()          # every word, no cap
index = {i + 3: w for w, i in keras.datasets.imdb.get_word_index().items()}   # Keras shifts indices by 3


def text(reviews, n_chars):
    out, size = [], 0
    for r in reviews:
        s = " ".join(index[i] for i in r if i >= 3) + "\n"          # drop the start and padding markers
        out.append(s)
        size += len(s)
        if size >= n_chars:
            break
    return "".join(out)[:n_chars]


train_text, val_text = text(x_train, 3_000_000), text(x_test, 300_000)
print(repr(train_text[:300]))
counts = pd.Series(list(train_text)).value_counts()
chars = sorted(counts[counts >= 50].index)                 # rare characters share one id
char_id = {c: i + 1 for i, c in enumerate(chars)}          # 0 = rare
V = len(chars) + 1
print("distinct characters kept:", len(chars), "| vocabulary incl. 'rare':", V)


def windows(t, length=100):
    ids = np.array([char_id.get(c, 0) for c in t], dtype="int32")
    n = (len(ids) - 1) // length
    return ids[:n * length].reshape(n, length), ids[1:n * length + 1].reshape(n, length)   # target = next char


Xtr, Ytr = windows(train_text)
Xva, Yva = windows(val_text)
print("training windows:", Xtr.shape, "validation windows:", Xva.shape)

In [ ]:
def char_model(kind):
    layers = [keras.Input(shape=(100,)), keras.layers.Embedding(V, 32)]
    if kind == "1 layer, 128 nodes":
        layers += [keras.layers.LSTM(128, return_sequences=True)]
    elif kind == "2 stacked layers, 128 + 128 nodes":
        layers += [keras.layers.LSTM(128, return_sequences=True), keras.layers.LSTM(128, return_sequences=True)]
    else:                                                   # one wide layer, about as many parameters as the stack
        layers += [keras.layers.LSTM(215, return_sequences=True)]
    layers += [keras.layers.Dense(V, activation="softmax")]  # a prediction at every time step
    return keras.Sequential(layers)


KINDS = ["1 layer, 128 nodes", "2 stacked layers, 128 + 128 nodes", "1 wide layer, 215 nodes"]
for k in KINDS:
    m = char_model(k)
    print(f"{k:36} recurrent parameters {sum(l.count_params() for l in m.layers[1:-1]):>8,}"
          f"   total {m.count_params():>8,}")

In [ ]:
EPOCHS, SEEDS = 10, [1, 2]
hist = []
for k in KINDS:
    for s in SEEDS:
        keras.utils.set_random_seed(s)
        m = char_model(k)
        m.compile(optimizer=keras.optimizers.Adam(2e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        h = m.fit(Xtr, Ytr, validation_data=(Xva, Yva), epochs=EPOCHS, batch_size=128, verbose=0)
        for e in range(EPOCHS):
            hist.append(dict(model=k, seed=s, epoch=e + 1, loss=h.history["loss"][e], val_loss=h.history["val_loss"][e],
                             val_accuracy=h.history["val_accuracy"][e]))
        print(f"{k:36} seed {s}: val loss {h.history['val_loss'][-1]:.4f}, val acc {h.history['val_accuracy'][-1]:.4f}")
hist = pd.DataFrame(hist)
hist.round(5).to_csv(DATA / "char_lm_history.csv", index=False)

In [ ]:
last = hist[hist.epoch == EPOCHS].groupby("model", sort=False)[["loss", "val_loss", "val_accuracy"]].mean()
print(last.round(4))
best = hist.groupby(["model", "seed"], sort=False).val_loss.min().groupby("model", sort=False).mean()
print("best validation loss (mean of seeds):\n", best.round(4))